# IFCE Campus Tauá - Tecnologia em ADS
## Banco de Dados Não-Relacionais (ADS16)
### Aula 04: Modelo Chave-Valor & Arquitetura em Memória com Redis
**Docente:** Prof. Me. Reginaldo Pereira Fernandes  
**Carga Horária:** 2h (120 minutos) | **Semestre:** 2026.2

---
### 🎯 Objetivos do Laboratório Prático
1. Compreender a arquitetura em memória e os trade-offs de volatilidade da RAM.
2. Configurar e testar persistência em disco (**RDB Snapshotting** e **AOF**).
3. Praticar o padrão **Cache-Aside** aplicado a um problema real: **Cache de Consultas de Busca** de catálogo.
4. Dominar as variações avançadas dos comandos `SET` (`EX`, `PX`, `NX`, `XX`, `KEEPTTL`, `GET`, `MSET`) e `GET` (`MGET`, `GETDEL`, `GETEX`, `GETRANGE`, `STRLEN`).
5. Explorar as 5 estruturas de dados nativas do Redis (Strings, Hashes, Lists, Sets, Sorted Sets).
6. Conectar o Redis com Inteligência Artificial: **LangCache** e Memória de Agentes.

## 0. Conexão com o Servidor Redis
> **Dica:** Se estiver no Windows, certifique-se de que o Redis está ativo no WSL (`sudo service redis-server start`) ou em container Docker.

In [ ]:
import redis
import json
import time
import random
import matplotlib.pyplot as plt
import numpy as np

# Conexao decodificada para strings UTF-8
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
print("PING Redis:", r.ping())

## 1. Problema Real Inicial: Cache de Termos de Busca (E-commerce)
Em e-commerces, consultas a bancos relacionais ou de documentos com filtros e texto completo demoram de 150ms a 300ms.  
Com o **Cache-Aside no Redis**, a primeira busca (**Cache Miss**) grava a resposta serializada na memória RAM com TTL de 120s, e as buscas subsequentes (**Cache Hit**) respondem em menos de 1 ms!

In [ ]:
def buscar_produtos_com_cache(termo: str, ttl_segundos: int = 120):
    chave = f"busca:catalogo:{termo.strip().lower().replace(" ", "_")}"
    inicio = time.perf_counter()
    dado = r.get(chave)
    
    if dado:
        duracao_ms = (time.perf_counter() - inicio) * 1000
        ttl = r.ttl(chave)
        print(f"⚡ [CACHE HIT] Termo: {termo} | Latencia: {duracao_ms:.3f} ms | TTL restante: {ttl}s")
        r.zincrby("ranking:buscas", 1, termo)
        return json.loads(dado), duracao_ms, "HIT"
    else:
        print(f"🐢 [CACHE MISS] Termo: {termo} -> Consultando banco em disco...")
        time.sleep(0.2) # Simula 200ms de disco/SQL
        produtos = [{"id": 101, "nome": f"Item {termo}", "preco": 299.90}]
        r.set(chave, json.dumps(produtos), ex=ttl_segundos)
        duracao_ms = (time.perf_counter() - inicio) * 1000
        print(f"💾 Gravado no Redis com TTL={ttl_segundos}s | Latencia: {duracao_ms:.2f} ms")
        r.zincrby("ranking:buscas", 1, termo)
        return produtos, duracao_ms, "MISS"

# Executando 1ª consulta (Miss) e 2ª consulta (Hit)
buscar_produtos_com_cache("notebook dell")
buscar_produtos_com_cache("notebook dell")

## 2. Variações Avançadas dos Comandos SET e GET
Explore as opções atômicas mais poderosas do Redis:

In [ ]:
# 1. SET NX (Trava Distribuida / Not Exists)
trava_1 = r.set("lock:estoque:101", "worker_A", nx=True, ex=10)
trava_2 = r.set("lock:estoque:101", "worker_B", nx=True, ex=10)
print(f"Worker A obteve trava? {trava_1} | Worker B concorrente: {trava_2}")
r.delete("lock:estoque:101")

# 2. SET com KEEPTTL
r.set("sessao:user_1", "dados_v1", ex=60)
print("TTL antes:", r.ttl("sessao:user_1"))
r.set("sessao:user_1", "dados_v2_atualizados", keepttl=True)
print("TTL apos KEEPTTL:", r.ttl("sessao:user_1"))

# 3. GETDEL (Token de uso unico)
r.set("token:reset:user_42", "TOKEN_SECRETO_99", ex=30)
token_consumido = r.getdel("token:reset:user_42")
print("Token consumido:", token_consumido, "| Consulta subsequente:", r.get("token:reset:user_42"))

# 4. MSET e MGET (Multiplas operacoes em lote)
r.mset({"cotacao:usd": "5.45", "cotacao:eur": "5.95"})
print("MGET:", r.mget(["cotacao:usd", "cotacao:eur"]))

## 3. Arquitetura em Memória e Persistência em Disco (RDB e AOF)
Como o Redis protege os dados da volatilidade da RAM?

In [ ]:
info_p = r.info("persistence")
print("Status do ultimo BGSAVE:", info_p["rdb_last_bgsave_status"])
print("Mudancas desde ultimo snapshot:", info_p["rdb_changes_since_last_save"])
print("AOF habilitado?", "Sim" if info_p.get("aof_enabled") == 1 else "Nao (Apenas RDB)")

# Dispara BGSAVE nao-bloqueante
try:
    print("Disparando BGSAVE:", r.bgsave())
except Exception as e:
    print("BGSAVE status:", e)

## 4. As 5 Estruturas Nativas em Problemas Reais

In [ ]:
# Hashes (Perfil)
r.hset("user:42", mapping={"nome": "Maria", "creditos": 150})
r.hincrby("user:42", "creditos", 50)
print("Hash User:", r.hgetall("user:42"))

# Lists (Historico capped)
r.delete("historico:buscas")
for item in ["notebook", "mouse", "teclado", "monitor", "fone", "webcam"]:
    r.lpush("historico:buscas", item)
    r.ltrim("historico:buscas", 0, 4)
print("Ultimas 5 buscas:", r.lrange("historico:buscas", 0, -1))

# Sets (Tags em comum)
r.delete("tags:A", "tags:B")
r.sadd("tags:A", "ti", "dev", "bivolt")
r.sadd("tags:B", "eletro", "dev", "bivolt")
print("Tags em comum (SINTER):", r.sinter("tags:A", "tags:B"))

# Sorted Sets (Ranking)
r.delete("ranking:buscas")
r.zadd("ranking:buscas", {"notebook": 50, "smartphone": 80, "fone": 30})
r.zincrby("ranking:buscas", 40, "notebook")
print("Top 2 mais buscados:", r.zrevrange("ranking:buscas", 0, 1, withscores=True))

## 5. Modernização com IA: LangCache e Memória de Agentes
> LLMs são *stateless*. Quase 40-80% dos prompts em empresas são repetidos ou similares. O Redis intercepta essas chamadas respondendo em < 1 ms!

In [ ]:
prompt = "Como funciona persistencia RDB no Redis?"
chave_prompt = f"langcache:{hash(prompt)}"

resp_cache = r.get(chave_prompt)
if resp_cache:
    print("⚡ [LANGCACHE HIT] Latencia: < 1 ms | Resposta:", resp_cache)
else:
    print("🧠 [LLM COLD CALL] Computando resposta com modelo remoto...")
    resposta = "RDB tira fotos pontuais da memoria e salva em disco via BGSAVE."
    r.set(chave_prompt, resposta, ex=3600)
    print("💾 Resposta armazenada no LangCache!")